# HuggingFace Pipeline Use

1. 주제를 선정한다.
2. 사전 학습(또는 전이 학습된) 모델을 HuggingFace로부터 Pipeline을 이용해 로드해 온다.
3. 내가 원하는 주제에 부합하는 시스템이 동작하도록 한다.
- console 에서 진행 OK
- streamlit 구현 OK

### TRPG 배경 설정 스토리 만들기

- 텍스트 생성으로 키워드만 입력하면 만들 수 있게
- 생성된 상세한 내용은 플레이어는 몰라야 하기 때문에 요약본으로 전달
- NER 을 이용해서 배경 속 지명이나 인물의 이름을 할당

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

##### 4비트 양자화
https://ariz1623.tistory.com/354 : 양자화가 한국어에 미치는 영향에 대한 블로그 정보글

- 가중치가 보통 16비트 or 32비트 인데 이를 8비트나 4비트로 줄이는 것 -> 모델 경량화에 사용
- 당연히 성능 저하가 있을 수 밖에 없으나 모델 경량화를 해야만 하는 상황이라면 사용.

In [ ]:
# !pip install bitsandbytes accelerate

   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   -------- ------------------------------- 11.8/55.4 MB 61.4 MB/s eta 0:00:01
   ------------------ --------------------- 25.2/55.4 MB 63.7 MB/s eta 0:00:01
   ---------------------- ----------------- 31.2/55.4 MB 50.7 MB/s eta 0:00:01
   ---------------------------- ----------- 40.1/55.4 MB 49.0 MB/s eta 0:00:01
   -------------------------------- ------- 45.6/55.4 MB 43.3 MB/s eta 0:00:01
   ---------------------------------------  55.3/55.4 MB 45.7 MB/s eta 0:00:01
   ---------------------------------------- 55.4/55.4 MB 42.0 MB/s  0:00:01

   ---------------------------------------- 0/2 [bitsandbytes]
   ---------------------------------------- 0/2 [bitsandbytes]
   ---------------------------------------- 0/2 [bitsandbytes]
   ---------------------------------------- 0/2 [bitsandbytes]
   ---------------------------------------- 0/2 [bitsandbytes]
   ---------------------------------------- 0/2 [bitsandbytes]
 

In [2]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 모델 사용

- Bllossom/llama-3.2-Korean-Bllossom-3B
- instruction tuning이 진행된 모델 -> 프롬프트 엔지니어링 방식으로 사용 가능

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=bnb_config 
)

`torch_dtype` is deprecated! Use `dtype` instead!
Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
Loading weights: 100%|██████████| 254/254 [00:15<00:00, 16.17it/s, Materializing param=model.norm.weight]                               


In [12]:
# 키워드 넣으면 생성해주는 함수로 만들기
def create_trpg_world(keywords):
    instruction = f"{keywords} 이 키워드들을 사용해서 trpg에 사용할 세계관을 짜줘."

    messages = [
        {"role": "system", "content": "당신은 한국인 TRPG 게임 마스터입니다. 한국어 위주로만 사용하고, 듣기만 해도 배경이 그려지는 듯하게 설명해주세요."},
        {"role": "user", "content": f"{instruction}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True  # 딕셔너리 형태로 반환 강제
    ).to(model.device)

    terminators = [
        tokenizer.convert_tokens_to_ids("<|end_of_text|>"),
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=512, 
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.55,
        top_p=0.8,
        repetition_penalty=1.2 # 같은 말 반복 방지
    )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)


In [13]:
# 2. 실제 사용 예시
my_keywords = "비행 함선, 잊혀진 마법, 폭풍의 눈"
result = create_trpg_world(my_keywords)

print(f"입력 키워드: {my_keywords}")
print(f"생성된 세계관:\n{result}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


입력 키워드: 비행 함선, 잊혀진 마법, 폭풍의 눈
생성된 세계관:
한국의 전통적인 동화 "아나키"를 바탕으로 한 비행함선과 잊겨진 마법, 폭풍의 눈이라는 키워드를 사용한 세계관을 짚겠습니다.

**세계 이름:** 아나키의 무성 (Anaeki's Mysterious)

**배경:** 아나키는 한국의 전통적인 동화를 바탕으로 한 세계관입니다. 이 세계에서는 인간들이 일상생활에서 큰 도움이 되지 않는 마법을 가지고 있습니다. 하지만 이 마법들은 자주 잊혀집니다. 예를 들어, jemand이 자신의 집안의 물건을 보일 수 있지만, 그의 집안의 모든 물건을 모두 잃어버리는 경우가 많습니다.

**비행함선:** 이 세계에는 다양한 비행함선을 가지고 있습니다. 이러한 비행함선들은 일반적으로 특정 지역이나 국가에 속합니다. 예를 들어, 북쪽의 아라리아 지방은 많은 비행함선을 가지고 있으며, 남쪽의 카르마니아 지방은 거의 keine 비행함선을 가지고 있습니다.

**잫어진 마법:** 이 세계에서는 다양한 잫어진 마법이 존재합니다. 예를 들어:

* **수리소리(수리소리)**: 사람들의 마음을 조종하는 마법입니다.
* **여미자루(여미자루)**: 사람들의 감정을 조절하는 마법입니다.
* **뿌리시(뿌리시)**: 자연 환경을 변형시키는 마법입니다.

이러한魔法들은 자주 잊혀져서, 사람들에게 효과를 주는 것이 어려울 수 있습니다.

**폭风의 눈:** 이 세계에서는 폭풍의 눈을 가진 인물들이 존재합니다. 이러한 인물들은 매우 강력하며, 다른 사람들과의 대립을 유발할 수도 있습니다. 예를 들어:

* **스카이파이어(Skyfire)**: 폭풍의 눈을 가진 인물입니다. 그는 항공우주국(Aerius)의 최고 장교 중 하나입니다.
* **하늘바위(Hauntree)**: 폭풍의 눈을 가진 인물입니다. 그녀는 북쪽의 아라리아 지방의 여왕입니다.

**사용법:** 이 worlds를 사용하기 위해서는 다음과 같은 방법을 사용하세요:

1. **플레이어**: 플레이어가 자신의 character와 